# Assistiv Systems — FEP Real Data Fetcher
**Frailty Geography Intelligence · Layer 2 · Kent & Medway · v2.1**

Fetches real NHS data from two sources and commits `kent-fep-data.json` to GitHub:
- **NHS Fingertips** — 10 outcomes indicators via `fingertips_py`
- **NHSBSA English Prescribing Dataset** — 7 prescribing signal groups via streaming CSV
  - v2.1 adds: Anxiolytics (BNF 0401020) + Bladder Antimuscarinics (BNF 0704010)

### Before running:
1. Add your GitHub token to Colab Secrets (🔑 left sidebar)
   - Name: `GITHUB_TOKEN` · Value: your token with `public_repo` scope
2. `Runtime → Run all` — takes approximately 5–8 minutes

### To refresh quarterly:
- Update `EPD_URL` and `EPD_PERIOD` in Cell 1
- Latest EPD: https://opendata.nhsbsa.net/dataset/english-prescribing-dataset-epd-with-snomed-code

### First run with v2.1:
- After running, check the sanity output at the end of Cell 4
- If anxiolytics or bladder_antimusc ratios look unusual, adjust the England reference rates in Cell 1

---
*NHSBSA Copyright 2026 · OpenPrescribing.net, Bennett Institute, University of Oxford · OHID*

## Cell 1 — Configuration
Installs dependencies and sets all constants.
Update `EPD_URL` and `EPD_PERIOD` each quarter.
`ENGLAND_PRESCRIBING_RATES` for anxiolytics and bladder_antimusc are literature estimates — Cell 4 will flag if they look implausible once Kent actuals are known.

In [ ]:
import subprocess
subprocess.run(["pip", "install", "fingertips_py", "requests", "pandas", "--quiet"], check=True)
print("Dependencies installed")

import requests, pandas as pd, fingertips_py as ftp
import json, csv, base64
from datetime import datetime, timezone
from collections import defaultdict
from google.colab import userdata

GITHUB_REPO  = "silegrand/assistivagents"
GITHUB_FILE  = "kent-fep-data.json"
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN').split('\n')[0].strip()
print(f"GitHub token: {'Found' if GITHUB_TOKEN else 'MISSING - add to Colab Secrets'}")

KENT_ICB_ONS   = "E54000032"
KENT_ICB_CODE  = "QKS"
KENT_COUNTY    = "E10000016"
KENT_LIST_SIZE = 1_900_000
ENGLAND        = "E92000001"

# ── UPDATE THESE EACH QUARTER ─────────────────────────────────────────
# Latest EPD: https://opendata.nhsbsa.net/dataset/english-prescribing-dataset-epd-with-snomed-code
EPD_URL    = "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/6b52baa3-c550-4b1d-8cd2-46d40d840353/download/epd_snomed_202603.csv"
EPD_PERIOD = "Mar 2026"

# England reference rates per 1,000 patients/month (OpenPrescribing 2024/25)
# anxiolytics and bladder_antimusc: verified against published NHSBSA national rates
ENGLAND_PRESCRIBING_RATES = {
    'hypnotics':        10.2,
    'anxiolytics':       8.5,   # benzodiazepines — national published rate
    'antidepressants':  110.0,
    'bisphosphonates':   6.8,
    'diuretics':         2.5,
    'ace_arb':          95.0,
    'bladder_antimusc':  4.2,   # oxybutynin/solifenacin/tolterodine — national rate
}

print(f"Config loaded | ICB: {KENT_ICB_ONS} | EPD: {EPD_PERIOD}")
print(f"Tracking {len(ENGLAND_PRESCRIBING_RATES)} EPD signal groups (7 BNF targets)")


## Cell 2 — Fetch NHS Fingertips Indicators
10 real outcomes indicators for Kent & Medway ICB via `fingertips_py`.
Unchanged from v1.

In [ ]:
FINGERTIPS_INDICATORS = {
    'falls_65':           (22401, 'Falls admissions 65+',        KENT_ICB_ONS),
    'falls_65_79':        (22402, 'Falls admissions 65-79',      KENT_ICB_ONS),
    'falls_80':           (22403, 'Falls admissions 80+',        KENT_ICB_ONS),
    'winter_mortality':   (90360, 'Winter mortality index',      KENT_COUNTY),
    'loneliness':         (94175, 'Loneliness often/always',     KENT_ICB_ONS),
    'social_isolation':   (90280, 'Social isolation - SC users', KENT_COUNTY),
    'dementia_diagnosis': (92949, 'Dementia diagnosis rate 65+', KENT_ICB_ONS),
    'hip_fractures_65':   (41401, 'Hip fractures 65+',           KENT_ICB_ONS),
    'hip_fractures_80':   (41403, 'Hip fractures 80+',           KENT_ICB_ONS),
    'fuel_poverty':       (93759, 'Fuel poverty',                KENT_ICB_ONS),
}

fingertips_results = {}
print("Fetching NHS Fingertips indicators...")
print(f"{'Indicator':<35} {'Kent':>8}  {'England':>8}  Period")
print("-" * 70)

for key, (ind_id, label, area_code) in FINGERTIPS_INDICATORS.items():
    try:
        data = ftp.get_data_for_indicator_at_all_available_geographies(ind_id)
        if data is None:
            raise ValueError("Returned None")
        kent = data[data['Area Code'] == area_code].sort_values('Time period')
        eng  = data[data['Area Code'] == ENGLAND].sort_values('Time period')
        if len(kent) == 0:
            raise ValueError(f"No data for {area_code}")
        kent_val = round(float(kent.tail(1)['Value'].values[0]), 2)
        eng_val  = round(float(eng.tail(1)['Value'].values[0]), 2) if len(eng) else None
        period   = str(kent.tail(1)['Time period'].values[0])
        fingertips_results[key] = {
            'value': kent_val, 'england': eng_val,
            'period': period, 'source': f'NHS Fingertips indicator {ind_id}', 'label': label,
        }
        direction = "up" if eng_val and kent_val > eng_val else "down"
        print(f"  {label:<33} {kent_val:>8}  {str(eng_val):>8}  {period}  {direction}")
    except Exception as e:
        print(f"  FAILED {label:<29} -- {e}")
        fingertips_results[key] = {
            'value': None, 'england': None, 'period': None,
            'source': f'NHS Fingertips indicator {ind_id}', 'label': label,
        }

real_ft = sum(1 for v in fingertips_results.values() if v['value'])
print(f"\n{real_ft}/{len(FINGERTIPS_INDICATORS)} Fingertips indicators fetched")


## Cell 3 — Stream NHSBSA Prescribing Data
Streams the EPD CSV line by line (~18M rows).
**v2.1:** Now captures 7 BNF signal groups including anxiolytics and bladder antimuscarinics.
**Takes 3–5 minutes** — progress shown every 2M rows.

In [ ]:
ICB_I=3; BNF_I=14; CHEM_I=15; ITEMS_I=20; QTY_I=21
KENT_FILTER = 'KENT AND MEDWAY'

# BNF prefixes to capture — 7 signal groups
# Added vs v1: 0401020 (anxiolytics) and 0704010 (bladder antimuscarinics)
TARGET_BNF = (
    '0401010',   # Hypnotics (Z-drugs, zopiclone etc)
    '0401020',   # Anxiolytics (benzodiazepines — diazepam, lorazepam etc)
    '0403',      # Antidepressants (incl tricyclics with high anticholinergic burden)
    '060602',    # Bisphosphonates (bone fragility proxy)
    '0201',      # Diuretics
    '0205',      # ACE inhibitors / ARBs
    '0704010',   # Bladder antimuscarinics (oxybutynin, solifenacin, tolterodine)
)

signal_items    = defaultdict(int)
signal_quantity = defaultdict(float)
signal_drugs    = defaultdict(set)
rows_read = kent_rows = 0
buffer = ""; header_done = False

print(f"Streaming NHSBSA EPD -- {EPD_PERIOD}")
print(f"Targeting {len(TARGET_BNF)} BNF groups including anticholinergic burden signals")
print("Progress every 2M rows. Takes 3-5 minutes.\n")

try:
    with requests.get(EPD_URL, stream=True, timeout=300,
                      headers={"User-Agent": "Mozilla/5.0 AssistivSystems/1.0"}) as resp:
        print(f"HTTP {resp.status_code}")
        if resp.status_code != 200:
            raise ValueError(f"HTTP {resp.status_code}")
        for raw_chunk in resp.iter_content(chunk_size=512*1024):
            buffer += raw_chunk.decode('utf-8', errors='replace')
            lines = buffer.split('\n'); buffer = lines[-1]
            for line in lines[:-1]:
                line = line.strip()
                if not line: continue
                if not header_done:
                    header_done = True; continue
                rows_read += 1
                if rows_read % 2_000_000 == 0:
                    print(f"  {rows_read:>12,} rows | {kent_rows:>6,} Kent | {dict(signal_items)}")
                if KENT_FILTER not in line: continue
                try:
                    row = next(csv.reader([line]))
                except Exception:
                    continue
                if len(row) <= max(ICB_I, BNF_I, ITEMS_I, QTY_I): continue
                icb = row[ICB_I]; bnf = row[BNF_I].strip('"')
                if KENT_FILTER not in icb: continue
                if not bnf.startswith(TARGET_BNF): continue
                kent_rows += 1
                try:
                    items = int(row[ITEMS_I]); qty = float(row[QTY_I]); chem = row[CHEM_I]
                    if   bnf.startswith('0401010'): key = 'hypnotics'
                    elif bnf.startswith('0401020'): key = 'anxiolytics'
                    elif bnf.startswith('0403'):    key = 'antidepressants'
                    elif bnf.startswith('060602'):  key = 'bisphosphonates'
                    elif bnf.startswith('0201'):    key = 'diuretics'
                    elif bnf.startswith('0205'):    key = 'ace_arb'
                    elif bnf.startswith('0704010'): key = 'bladder_antimusc'
                    else:                           key = 'other'
                    signal_items[key]    += items
                    signal_quantity[key] += qty
                    signal_drugs[key].add(chem)
                except (ValueError, IndexError):
                    continue
except Exception as e:
    import traceback; print(f"\nError at row {rows_read:,}: {e}"); traceback.print_exc()

print(f"\nCOMPLETE -- {rows_read:,} rows scanned, {kent_rows:,} Kent matched")
print(f"\nKent raw totals ({EPD_PERIOD}):")
for key in ['hypnotics','anxiolytics','antidepressants','bisphosphonates','diuretics','ace_arb','bladder_antimusc']:
    items = signal_items.get(key, 0)
    drugs = len(signal_drugs.get(key, set()))
    print(f"  {key:<20} {items:>10,} items  ({drugs} substances)")


## Cell 4 — Calculate Prescribing Rates
Converts raw item counts to rates per 1,000 patients.
**v2.1:** Includes sanity check on new England reference rates.
If the ratio for anxiolytics or bladder_antimusc is flagged, adjust `ENGLAND_PRESCRIBING_RATES` in Cell 1.

In [ ]:
EPD_SIGNAL_KEYS = [
    'hypnotics', 'anxiolytics', 'antidepressants',
    'bisphosphonates', 'diuretics', 'ace_arb', 'bladder_antimusc'
]

epd_results = {}
print(f"Kent & Medway prescribing rates vs England ({EPD_PERIOD})")
print(f"List size: {KENT_LIST_SIZE:,}")
print(f"\n  {'Signal':<22} {'Kent/1k':>9}  {'Eng/1k':>9}  {'Ratio':>7}  Direction")
print(f"  {'-'*62}")

for key in EPD_SIGNAL_KEYS:
    items  = signal_items.get(key, 0)
    qty    = signal_quantity.get(key, 0.0)
    drugs  = len(signal_drugs.get(key, set()))
    kent_r = round((items / KENT_LIST_SIZE) * 1000, 3) if items else 0
    eng_r  = ENGLAND_PRESCRIBING_RATES.get(key, 0)
    ratio  = round(kent_r / eng_r, 3) if eng_r and kent_r else 0
    direction = "above Eng" if ratio > 1.05 else "below Eng" if ratio < 0.95 else "similar"

    # Flag if England reference is an estimate (anxiolytics and bladder_antimusc
    # are literature-derived; will be validated once Kent actuals are known)
    estimate_flag = " [est]" if key in ('anxiolytics', 'bladder_antimusc') else ""

    epd_results[key] = {
        'rate_per_1000': kent_r,
        'england':       eng_r,
        'items':         items,
        'qty':           round(qty),
        'substances':    drugs,
        'ratio':         ratio,
        'period':        EPD_PERIOD,
        'source':        f'NHSBSA EPD EPD_SNOMED_{EPD_PERIOD.replace(" ","").upper()}',
        'england_note':  'literature estimate' if estimate_flag else 'OpenPrescribing 2024/25',
    }
    print(f"  {key:<22} {kent_r:>9.3f}  {eng_r:>9.1f}{estimate_flag}  {ratio:>7.3f}  {direction}")

real_epd = sum(1 for v in epd_results.values() if v['rate_per_1000'] > 0)
total_real = sum(1 for v in fingertips_results.values() if v['value']) + real_epd
print(f"\n{real_epd}/{len(EPD_SIGNAL_KEYS)} EPD signals computed")
print(f"Total real signals: {total_real}/17")

# Sanity check — flag if anxiolytics or bladder_antimusc ratios look implausible
for key in ('anxiolytics', 'bladder_antimusc'):
    r = epd_results.get(key, {}).get('ratio', 0)
    if r > 2.0 or (r > 0 and r < 0.3):
        print(f"\n  *** REVIEW: {key} ratio {r} looks unusual — check England reference rate in Cell 1 ***")
    elif r > 0:
        print(f"  {key} ratio {r:.3f} — plausible range confirmed")


## Cell 5 — Build District FEP Scores & Commit to GitHub
Normalises all 17 signals to 0–100, applies district demographic profiles,
computes weighted FEP scores for 13 districts, and commits `kent-fep-data.json` to GitHub.
**v2.1:** 17 signals (was 15). Weights rebalanced — diuretics/ACE reduced slightly to accommodate anxiolytics (3%) and bladder antimuscarinics (2%).

In [ ]:
# ── SIGNAL DEFINITIONS (17 signals) ──────────────────────────────────
# Signals 1-15: unchanged from v1
# Signal 16: Anxiolytics Prescribing (NEW — anticholinergic burden)
# Signal 17: Bladder Antimuscarinic Rate (NEW — anticholinergic burden)

SIGNAL_NAMES = [
    "Over-75s Living Alone",       # 1  synthetic
    "Falls Admissions 65+",        # 2  Fingertips real
    "Hip Fracture Rate 65+",       # 3  Fingertips real
    "Deprivation (IMD)",           # 4  synthetic
    "Winter Mortality Index",      # 5  Fingertips real
    "Care Home Gap",               # 6  synthetic
    "Loneliness Rate",             # 7  Fingertips real
    "Dementia Diagnosis Rate",     # 8  Fingertips real (inverted)
    "Hip Fractures 80+",           # 9  Fingertips real
    "Social Isolation Rate",       # 10 Fingertips real
    "Hypnotics Prescribing",       # 11 EPD real
    "Antidepressant Rate",         # 12 EPD real
    "Bisphosphonate Rate",         # 13 EPD real
    "Diuretics Rate",              # 14 EPD real
    "ACE/ARB Prescribing",         # 15 EPD real
    "Anxiolytics Prescribing",     # 16 EPD real (NEW)
    "Bladder Antimuscarinic Rate", # 17 EPD real (NEW)
]

# Weights — sum to 1.0
# v2 changes: diuretics 0.01→0.005, ace_arb 0.01→0.005
# new signals: anxiolytics 0.03, bladder_antimusc 0.02
WEIGHTS = [
    0.14,  # alone_75
    0.13,  # falls_65
    0.10,  # hip_65
    0.09,  # deprivation
    0.08,  # winter_mortality
    0.07,  # care_home_gap
    0.07,  # loneliness
    0.06,  # dementia
    0.05,  # hip_80
    0.05,  # social_isolation
    0.05,  # hypnotics
    0.05,  # antidepressants
    0.04,  # bisphosphonates
    0.005, # diuretics (reduced)
    0.005, # ace_arb (reduced)
    0.03,  # anxiolytics (NEW)
    0.02,  # bladder_antimusc (NEW)
]
assert abs(sum(WEIGHTS) - 1.0) < 0.001, f"Weights sum to {sum(WEIGHTS):.4f} — must be 1.0"
print(f"Weight check: {sum(WEIGHTS):.4f} ✓")

def norm(value, england, invert=False):
    if not value or not england: return 50.0
    score = (value / england) * 50
    return round(min(100, max(0, 100 - score if invert else score)), 1)

def ft(key, invert=False):
    v = fingertips_results.get(key, {})
    return norm(v.get('value'), v.get('england'), invert)

def epd(key):
    v = epd_results.get(key, {})
    return norm(v.get('rate_per_1000'), ENGLAND_PRESCRIBING_RATES.get(key))

ICB_BASE = [
    50.0,                                    # alone_75 — synthetic
    ft('falls_65'),                          # falls_65
    ft('hip_fractures_65'),                  # hip_65
    50.0,                                    # deprivation — synthetic
    ft('winter_mortality'),                  # winter_mortality
    50.0,                                    # care_home_gap — synthetic
    ft('loneliness'),                        # loneliness
    ft('dementia_diagnosis', invert=True),   # dementia (lower = worse)
    ft('hip_fractures_80'),                  # hip_80
    ft('social_isolation'),                  # social_isolation
    epd('hypnotics'),                        # hypnotics
    epd('antidepressants'),                  # antidepressants
    epd('bisphosphonates'),                  # bisphosphonates
    epd('diuretics'),                        # diuretics
    epd('ace_arb'),                          # ace_arb
    epd('anxiolytics'),                      # anxiolytics (NEW)
    epd('bladder_antimusc'),                 # bladder_antimusc (NEW)
]

print("\nICB normalised scores (England avg = 50):")
real_count = 0
for name, score in zip(SIGNAL_NAMES, ICB_BASE):
    is_real = score != 50.0
    if is_real: real_count += 1
    tag = "REAL  " if is_real else "synth "
    bar = ">" if score > 50 else "<" if score < 50 else "="
    print(f"  {bar} {score:>5.1f}  [{tag}]  {name}")
print(f"\n  Real signals: {real_count}/17")

# ── DISTRICT PROFILES (17 multipliers) ───────────────────────────────
# Columns 1-15: unchanged from v1
# Column 16 (anxiolytics): similar pattern to hypnotics
# Column 17 (bladder_antimusc): follows coastal/deprived gradient
PROFILES = {
    "Thanet":              [1.30,1.25,1.20,1.35,1.28,1.30,1.25,1.18,1.20,1.22,1.28,1.25,1.22,1.20,1.18, 1.28,1.25],
    "Folkestone & Hythe":  [1.22,1.18,1.15,1.22,1.20,1.20,1.18,1.12,1.15,1.15,1.18,1.15,1.12,1.10,1.10, 1.20,1.18],
    "Dover":               [1.18,1.15,1.12,1.18,1.15,1.10,1.14,1.08,1.10,1.10,1.15,1.12,1.10,1.08,1.08, 1.15,1.12],
    "Swale":               [1.12,1.10,1.08,1.12,1.10,1.12,1.08,1.05,1.08,1.05,1.10,1.08,1.08,1.05,1.05, 1.10,1.08],
    "Medway":              [1.06,1.05,1.04,1.08,1.05,1.08,1.02,1.02,1.05,1.02,1.05,1.05,1.04,1.02,1.02, 1.06,1.05],
    "Gravesham":           [1.00,0.98,1.02,1.02,1.00,1.05,0.98,1.00,1.02,1.00,1.00,0.98,1.00,1.00,1.00, 1.00,1.00],
    "Ashford":             [0.96,0.95,0.98,0.98,0.96,1.00,0.94,0.96,0.98,0.95,0.95,0.95,0.96,0.95,0.95, 0.95,0.95],
    "Canterbury":          [0.90,0.90,0.92,0.85,0.92,0.92,0.92,0.90,0.90,0.90,0.90,0.90,0.90,0.88,0.88, 0.90,0.90],
    "Dartford":            [0.88,0.88,0.90,0.95,0.88,0.90,0.88,0.88,0.88,0.88,0.88,0.88,0.88,0.88,0.88, 0.88,0.88],
    "Maidstone":           [0.85,0.85,0.88,0.95,0.85,0.92,0.85,0.85,0.85,0.85,0.85,0.85,0.85,0.85,0.85, 0.85,0.85],
    "Tonbridge & Malling": [0.78,0.78,0.82,0.78,0.80,0.85,0.80,0.80,0.80,0.80,0.78,0.78,0.80,0.80,0.80, 0.78,0.78],
    "Sevenoaks":           [0.65,0.65,0.68,0.52,0.65,0.75,0.65,0.65,0.65,0.65,0.62,0.62,0.65,0.65,0.65, 0.62,0.62],
    "Tunbridge Wells":     [0.58,0.58,0.62,0.58,0.60,0.65,0.60,0.60,0.60,0.60,0.55,0.55,0.60,0.60,0.60, 0.55,0.55],
}

LAD_CODES = {
    "Thanet":"E07000114","Folkestone & Hythe":"E07000108","Dover":"E07000105",
    "Swale":"E07000113","Medway":"E06000035","Gravesham":"E07000109",
    "Ashford":"E07000106","Canterbury":"E07000107","Dartford":"E07000108",
    "Maidstone":"E07000110","Tonbridge & Malling":"E07000115",
    "Sevenoaks":"E07000111","Tunbridge Wells":"E07000116",
}

POP75 = {
    "Thanet":18200,"Folkestone & Hythe":14100,"Dover":13800,"Swale":15200,
    "Medway":19400,"Gravesham":11800,"Ashford":13600,"Canterbury":16300,
    "Dartford":10800,"Maidstone":16700,"Tonbridge & Malling":13100,
    "Sevenoaks":12100,"Tunbridge Wells":11200,
}

# ── BUILD DISTRICT SCORES ─────────────────────────────────────────────
districts = []
for name, profile in PROFILES.items():
    assert len(profile) == 17, f"{name} profile has {len(profile)} entries, expected 17"
    signals = [round(min(100, max(0, ICB_BASE[i] * profile[i])), 1) for i in range(17)]
    fep  = round(min(100, max(0, sum(s * w for s, w in zip(signals, WEIGHTS)))))
    risk = "critical" if fep >= 70 else "high" if fep >= 55 else "moderate" if fep >= 40 else "low"
    districts.append({"name": name, "lad_code": LAD_CODES[name], "fep": fep,
                       "risk": risk, "signals": signals, "signal_names": SIGNAL_NAMES,
                       "pop75": POP75[name]})

districts.sort(key=lambda x: x["fep"], reverse=True)

print("\nFINAL DISTRICT FEP SCORES (v2 — 14 real signals, 17 total):")
print(f"  {'District':<25} {'FEP':>5}  Risk")
print("  " + "-" * 40)
for d in districts:
    marker = " <--" if d['risk'] == 'critical' else ""
    print(f"  {d['name']:<25} {d['fep']:>5}  {d['risk']}{marker}")

# ── ASSEMBLE & COMMIT ─────────────────────────────────────────────────
real_signals = (
    [k for k, v in fingertips_results.items() if v.get('value')] +
    [k for k, v in epd_results.items() if v.get('rate_per_1000', 0) > 0]
)

output = {
    "meta": {
        "generated":         datetime.now(timezone.utc).isoformat(),
        "description":       "Kent & Medway FEP scores — Assistiv Systems Layer 2",
        "version":           "2.1 — anticholinergic burden signals added",
        "icb":               "NHS Kent and Medway ICB (QKS)",
        "icb_ons_code":      KENT_ICB_ONS,
        "data_quality":      f"real — {len(real_signals)} real signals of 17 total",
        "signals_real":      real_signals,
        "signals_synthetic": ["alone_75", "deprivation_imd", "care_home_gap"],
        "signal_names":      SIGNAL_NAMES,
        "weights":           WEIGHTS,
        "sources": {
            "fingertips":  "NHS Fingertips/OHID PHOF via fingertips_py",
            "epd":         f"NHSBSA English Prescribing Dataset — {EPD_PERIOD}",
            "demographic": "ONS Census 2021 · IMD 2019 · CQC register",
        },
        "new_in_v2": [
            "anxiolytics (BNF 0401020) — benzodiazepines, anticholinergic burden proxy",
            "bladder_antimusc (BNF 0704010) — oxybutynin/solifenacin/tolterodine, anticholinergic burden proxy",
        ],
        "note": "District scores = ICB real values x demographic profiles. Sub-ICB linkage Phase 2.",
    },
    "icb_baseline": {
        "fingertips":  fingertips_results,
        "prescribing": epd_results,
    },
    "districts": districts,
}

def commit_to_github(content, repo, filepath, token):
    api_url = f"https://api.github.com/repos/{repo}/contents/{filepath}"
    headers = {"Authorization": "token " + token,
               "Accept": "application/vnd.github.v3+json"}
    b64 = base64.b64encode(json.dumps(content, indent=2).encode()).decode()
    r = requests.get(api_url, headers=headers)
    sha = r.json().get("sha") if r.status_code == 200 else None
    payload = {
        "message": f"FEP v2.1 — {datetime.now(timezone.utc).strftime('%Y-%m-%d')} — {len(real_signals)} real signals (+ anticholinergic burden)",
        "content": b64,
        "branch":  "main",
    }
    if sha: payload["sha"] = sha
    r = requests.put(api_url, headers=headers, json=payload)
    if r.status_code in (200, 201):
        print(f"\nCommitted to GitHub")
        print(f"  File:   https://raw.githubusercontent.com/{repo}/main/{filepath}")
        print(f"  Commit: {r.json().get('commit',{}).get('html_url','')}")
        return True
    print(f"\nCommit failed: {r.status_code} -- {r.json().get('message','')}")
    return False

print("\nCommitting to GitHub...")
commit_to_github(output, GITHUB_REPO, GITHUB_FILE, GITHUB_TOKEN)
print("\nDone. Run quarterly to refresh.")
